In [0]:
# Databricks notebook cell 1
# Goal:
# 1) Build a file inventory table for all *.txt under SRC_DIR (no collect/toPandas)
# 2) Add a shard_id for batch processing later

from pyspark.sql import functions as F

SRC_DIR = "abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/converted_raw_txts/"
OUT_FILES_TABLE = "tdis_dev_data_catalog.tdir.stageA_txt_files"   # new table
N_SHARDS = 200  # batch control: later we process shard_id in [0..N_SHARDS-1]

df_files = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.txt")
    .option("recursiveFileLookup", "true")
    .load(SRC_DIR)
    .select(
        F.col("path").alias("txt_path"),
        F.col("length").alias("byte_len"),
        F.col("modificationTime").alias("mtime")
    )
    .withColumn("txt_path_lc", F.lower(F.col("txt_path")))
    # stable shard_id: hash(path) % N_SHARDS
    .withColumn("shard_id", (F.pmod(F.xxhash64("txt_path"), F.lit(N_SHARDS))).cast("int"))
    .drop("txt_path_lc")
)

# Write inventory
(df_files.write
 .mode("overwrite")
 .format("delta")
 .option("overwriteSchema", "true")
 .saveAsTable(OUT_FILES_TABLE)
)

print("OUT_FILES_TABLE =", OUT_FILES_TABLE)
print("total_txt_files =", spark.table(OUT_FILES_TABLE).count())

display(
    spark.table(OUT_FILES_TABLE)
    .groupBy("shard_id")
    .count()
    .orderBy("shard_id")
    .limit(20)
)

In [0]:
# Databricks notebook cell 2 (Spark-native file gate; no SparkContext on workers)

import re
from pyspark.sql import functions as F
from pyspark.sql import types as T

FILES_TABLE = "tdis_dev_data_catalog.tdir.stageA_txt_files"
OUT_GATE_TABLE = "tdis_dev_data_catalog.tdir.stageB_file_gate"

SHARD_ID = 0          # <--- change each run
MAX_LINES = 2000      # per file head lines used for gate
HEAD_CHARS = 250      # store tiny preview for debug

TEXAS_ONLY = True
DISASTER_ONLY = True

# ---------- rules ----------
DOMAIN_KEYWORDS_CORE = [
    "disaster","hazard","hazard mitigation","flood","flooding","flash flood",
    "storm","severe storm","hurricane","tropical storm","tropical cyclone",
    "tornado","wildfire","winter storm","freeze","cold wave","evacuation",
    "emergency operations plan","eop","hazard mitigation plan","hmp",
    "after action report","aar","improvement plan","state flood plan","regional flood plan",
    "stormwater master plan","mitigation plan","power outage","electric reliability","grid",
    "critical infrastructure","resilience","resiliency","emergency management",
]
DOMAIN_KEYWORDS_SUPPORT = [
    "watershed","water quality","cwa 319","stormwater","runoff","nepa",
    "environmental impact statement","categorical exclusion","emergency declaration",
    "disaster declaration","fema","tdem","twdb","glo","hcfcd","emergency operations",
    "continuity of operations","coop","cog",
]
TEXAS_STRONG_TERMS = [
    "twdb","tdem","hcfcd","txdot","ercot",
    "texas commission on environmental quality","tceq","glo"
]
STRONG_DOMAIN_TERMS = [
    "flood","flooding","inundation","stormwater","watershed","runoff","drainage",
    "levee","dam","reservoir","rainfall","precipitation","hurricane","tornado","wildfire",
    "winter storm","drought","evacuation","emergency management","hazard mitigation","mitigation",
    "fema","water quality","groundwater","aquifer","wastewater","sewer","treatment plant",
    "erosion","sediment","superfund","epa","contamination",
    "critical infrastructure","power outage","grid reliability","telecommunications"
]
EVENT_CORE_TERMS = [
    "flood","flooding","flash flood","hurricane","tropical storm","tropical cyclone",
    "tornado","wildfire","winter storm","freeze","cold wave","drought",
    "heat wave","heatwave","storm surge"
]
EVENT_CONTEXT_TERMS = [
    "fema","evacuation","shelter","sheltering","warning","watch","advisory",
    "disaster declaration","emergency declaration","after action report","aar",
    "hazard mitigation plan","hmp","damage","loss","payout","recovery",
    "inundation","stormwater","watershed","runoff","rainfall","precipitation",
    "streamflow","surge","floodplain","nfip","flood insurance"
]
FILE_BLACKLIST_TERMS = [
    "touchdowns","receiving yards","rushing yards","3pt","box score",
    "staar","bullying","fluency checks","instructional coach","curriculum",
    "grades 2-4","grades 9-12","student achievement","campus meets or exceeds",
    "biography:","curriculum vitae","cv:",
    "roundtable:","chair:","comment:","location:","omni",
    "see fish and fisheries","fisheries, see"
]

def _clean_text(text: str) -> str:
    if not text:
        return ""
    t = re.sub(r"<image:[^>]+>", " ", text)
    t = re.sub(r"_+", "____", t)
    t = " ".join(t.split())
    return t

def _is_domain_relevant_text(text: str, core_min_hits: int = 1, support_min_hits: int = 4) -> bool:
    if not text:
        return False
    t = text.lower()
    if sum(1 for kw in DOMAIN_KEYWORDS_CORE if kw in t) >= core_min_hits:
        return True
    return sum(1 for kw in DOMAIN_KEYWORDS_SUPPORT if kw in t) >= support_min_hits

def _is_texas_related_strong(text: str) -> bool:
    if not text:
        return False
    t = text.lower()
    if re.search(r"\btexas\b", t):
        return True
    return any(k in t for k in TEXAS_STRONG_TERMS)

def _has_strong_domain_signal(text: str, min_hits: int = 1) -> bool:
    if not text:
        return False
    t = text.lower()
    hits = sum(1 for kw in STRONG_DOMAIN_TERMS if kw in t)
    return hits >= min_hits

def _disaster_gate_file(text: str) -> bool:
    if not text:
        return False
    t = text.lower()
    if any(b in t for b in FILE_BLACKLIST_TERMS):
        return False
    core_ok = any(k in t for k in EVENT_CORE_TERMS)
    ctx_ok = any(k in t for k in EVENT_CONTEXT_TERMS)
    return core_ok and ctx_ok

def _file_decision(raw_text: str, texas_only: bool, disaster_only: bool):
    text = _clean_text(raw_text)
    if not text:
        return (False, "EMPTY")
    if not _is_domain_relevant_text(text):
        return (False, "FAIL_domain_relevant")
    if texas_only and (not _is_texas_related_strong(text)):
        return (False, "FAIL_texas_strong")
    if not _has_strong_domain_signal(text, min_hits=1):
        return (False, "FAIL_strong_domain")
    if disaster_only and (not _disaster_gate_file(text)):
        return (False, "FAIL_disaster_gate_file")
    return (True, "PASS")

udf_decision = F.udf(
    lambda s: _file_decision(s, TEXAS_ONLY, DISASTER_ONLY),
    T.StructType([
        T.StructField("passed", T.BooleanType(), False),
        T.StructField("reason", T.StringType(), False),
    ])
)

# ---------- shard paths ----------
df_paths = (
    spark.table(FILES_TABLE)
    .where(F.col("shard_id") == F.lit(SHARD_ID))
    .select("txt_path", "byte_len", "mtime", "shard_id")
)

# collect only paths (small: ~149k/200 shards ≈ 750 each) -> safe
paths = [r["txt_path"] for r in df_paths.select("txt_path").collect()]
print("shard_paths =", len(paths))

# ---------- read shard files with spark reader (multi-path) ----------
df_lines = (
    spark.read.format("text")
    .load(paths)
    .withColumn("txt_path", F.input_file_name())
    .withColumn("line_no", F.monotonically_increasing_id())
)

# take first MAX_LINES lines per file
w = (
    __import__("pyspark").sql.window.Window
    .partitionBy("txt_path")
    .orderBy("line_no")
)
df_head = (
    df_lines
    .withColumn("rn", F.row_number().over(w))
    .where(F.col("rn") <= F.lit(MAX_LINES))
    .groupBy("txt_path")
    .agg(F.concat_ws("\n", F.collect_list(F.col("value"))).alias("raw_head"))
)

df_gate = (
    df_paths.join(df_head, on="txt_path", how="left")
    .withColumn("dec", udf_decision(F.col("raw_head")))
    .select(
        "txt_path","byte_len","mtime","shard_id",
        F.col("dec.passed").alias("passed"),
        F.col("dec.reason").alias("reason"),
        F.substring(F.col("raw_head"), 1, HEAD_CHARS).alias("head_250"),
    )
    .withColumn("run_shard_id", F.lit(SHARD_ID).cast("int"))
    .withColumn("max_lines", F.lit(MAX_LINES).cast("int"))
    .withColumn("texas_only", F.lit(TEXAS_ONLY))
    .withColumn("disaster_only", F.lit(DISASTER_ONLY))
)

(df_gate.write
 .mode("append")
 .format("delta")
 .saveAsTable(OUT_GATE_TABLE)
)

print("WROTE =", OUT_GATE_TABLE)
display(
    df_gate.groupBy("passed","reason").count().orderBy(F.desc("count")).limit(30)
)

In [0]:
# Databricks notebook cell 3
# Goal:
# - Chunk ONLY passed files from stageB_file_gate (one shard)
# - Sentence split by (. ! ? ;)
# - Merge into chunks with <= MAX_LEN chars
# - Write to Delta (append): txt_id, txt_path, chunk_id, chunk_text, shard_id

import re
from pyspark.sql import functions as F
from pyspark.sql import types as T

GATE_TABLE = "tdis_dev_data_catalog.tdir.stageB_file_gate"
OUT_CHUNK_TABLE = "tdis_dev_data_catalog.tdir.stageC_chunks"

SHARD_ID = 0          # must match the shard you gated
MAX_LINES = 200000    # max lines per file to read (safety)
MAX_LEN = 1000        # chunk length

# ---------- select passed paths ----------
df_pass = (
    spark.table(GATE_TABLE)
    .where((F.col("shard_id") == F.lit(SHARD_ID)) & (F.col("passed") == F.lit(True)))
    .select("txt_path")
    .distinct()
)

pass_paths = [r["txt_path"] for r in df_pass.collect()]
print("pass_paths =", len(pass_paths))

# ---------- read all passed files (Spark-native) ----------
df_lines = (
    spark.read.format("text")
    .load(pass_paths)
    .withColumn("txt_path", F.input_file_name())
    .select("txt_path", F.col("value").cast("string").alias("line"))
)

# optional: limit per file lines (avoid insane files)
w = __import__("pyspark").sql.window.Window.partitionBy("txt_path").orderBy(F.monotonically_increasing_id())
df_lines = (
    df_lines
    .withColumn("rn", F.row_number().over(w))
    .where(F.col("rn") <= F.lit(MAX_LINES))
    .drop("rn")
)

# ---------- assemble full text per file ----------
df_text = (
    df_lines
    .groupBy("txt_path")
    .agg(F.concat_ws("\n", F.collect_list("line")).alias("raw_text"))
)

# ---------- UDF: text -> list[chunks] ----------
def split_sentences(t: str):
    # English comments only
    t = " ".join((t or "").split())
    if not t:
        return []
    parts = re.split(r"(?<=[\.\!\?\;])\s+", t)
    return [p.strip() for p in parts if p and p.strip()]

def sentences_to_chunks(sents, max_len: int):
    # English comments only
    out = []
    cur = ""
    for s in sents:
        if not cur:
            if len(s) <= max_len:
                cur = s
            else:
                for i in range(0, len(s), max_len):
                    piece = s[i:i+max_len]
                    if len(piece) == max_len:
                        out.append(piece)
                    else:
                        cur = piece
            continue

        cand = cur + " " + s
        if len(cand) <= max_len:
            cur = cand
        else:
            out.append(cur)
            cur = ""
            if len(s) <= max_len:
                cur = s
            else:
                for i in range(0, len(s), max_len):
                    piece = s[i:i+max_len]
                    if len(piece) == max_len:
                        out.append(piece)
                    else:
                        cur = piece
    if cur:
        out.append(cur)
    return out

def text_to_chunks(t: str):
    # English comments only
    sents = split_sentences(t)
    return sentences_to_chunks(sents, MAX_LEN)

udf_chunks = F.udf(text_to_chunks, T.ArrayType(T.StringType()))

# ---------- build chunk rows ----------
df_chunks = (
    df_text
    .withColumn("txt_id", F.sha1(F.col("txt_path")))
    .withColumn("chunks", udf_chunks(F.col("raw_text")))
    .select("txt_id", "txt_path", F.posexplode("chunks").alias("chunk_idx", "chunk_text"))
    .withColumn("chunk_id", F.concat_ws("_", F.col("txt_id"), F.col("chunk_idx").cast("string")))
    .withColumn("shard_id", F.lit(SHARD_ID).cast("int"))
    .select("txt_id", "txt_path", "chunk_id", "chunk_text", "shard_id")
)

print("chunk_rows =", df_chunks.count())
display(df_chunks.limit(10))

# ---------- write ----------
(df_chunks.write
 .mode("append")
 .format("delta")
 .saveAsTable(OUT_CHUNK_TABLE)
)

print("WROTE =", OUT_CHUNK_TABLE)

In [0]:
# Databricks notebook cell 4
# Stage1: LLM decide KEEP/DROP for chunks (one shard per run), write KEEP only.

import re
import time
import yaml
from pyspark.sql import functions as F
from openai import OpenAI

# ---------- tables ----------
SRC_CHUNK_TABLE = "tdis_dev_data_catalog.tdir.stageC_chunks"
OUT_KEEP_TABLE  = "tdis_dev_data_catalog.tdir.stageD_llm_kept"

SHARD_ID = 0   # <--- set each run

# ---------- endpoint ----------
MODEL_NAME = "databricks-gpt-oss-20b"
BASE_URL   = "https://adb-3300405005568038.18.azuredatabricks.net/serving-endpoints"
CONFIG_PATH = "../config.yaml"
TOKEN_KEY   = "DATABRICKS_TOKEN"

# ---------- tuning ----------
BATCH = 32
MAX_OUT = 300
TEMPERATURE = 0.0
N_PARTITIONS = 16

# ---------- disable mlflow autolog ----------
def disable_mlflow_autolog():
    # English comments only
    try:
        import mlflow
        try:
            import mlflow.openai
            mlflow.openai.autolog(disable=True)
        except Exception:
            pass
        try:
            mlflow.autolog(disable=True)
        except Exception:
            pass
    except Exception:
        pass

disable_mlflow_autolog()

# ---------- prompt + parsing ----------
def build_sys_prompt() -> str:
    return (
        "For each chunk, output exactly one line: <i>\\tKEEP or <i>\\tDROP (i starts from 1).\n"
        "DROP if mainly ads/marketing/CTA, contact info, navigation/boilerplate, or mostly links/listing with little content.\n"
        "Otherwise KEEP.\n"
        "No extra text. Do not output reasoning."
    )

def build_user_prompt(items):
    parts = []
    for it in items:
        i = int(it["i"])
        txt = str(it["text"])
        parts.append(f"[{i}]\n{txt}")
    return "\n\n".join(parts)

def parse_decisions(text_out: str):
    pairs = []
    for ln in (text_out or "").splitlines():
        m = re.match(r"^\s*(\d+)\s*(?:\\t|\t|:|\s)\s*(KEEP|DROP)\s*$", ln, flags=re.I)
        if m:
            pairs.append((int(m.group(1)), m.group(2).lower()))
    pairs.sort(key=lambda x: x[0])
    return [d for _, d in pairs]

def extract_text_from_content(content):
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        for part in content:
            if isinstance(part, dict) and part.get("type") in ("text", "output_text"):
                if isinstance(part.get("text"), str) and part["text"].strip():
                    return part["text"]
        for part in content:
            if isinstance(part, dict) and part.get("summary"):
                summ = part.get("summary")
                if isinstance(summ, list):
                    for s in summ:
                        if isinstance(s, dict) and s.get("type") in ("summary_text", "text"):
                            if isinstance(s.get("text"), str) and s["text"].strip():
                                return s["text"]
    return str(content)

def load_token(config_path: str, token_key: str) -> str:
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    return cfg[token_key]

# ---------- partition worker ----------
def llm_filter_partition(rows_iter):
    # English comments only
    disable_mlflow_autolog()

    token = load_token(CONFIG_PATH, TOKEN_KEY)
    client = OpenAI(api_key=token, base_url=BASE_URL)

    sys_prompt = build_sys_prompt()

    buf = []
    for r in rows_iter:
        buf.append(r)
        if len(buf) < BATCH:
            continue

        items = [{"i": j+1, "text": buf[j]["chunk_text"]} for j in range(len(buf))]
        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": build_user_prompt(items)},
        ]

        last_err = None
        out_text = ""
        for _ in range(6):
            try:
                resp = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=messages,
                    max_tokens=MAX_OUT,
                    temperature=TEMPERATURE,
                )
                out_text = extract_text_from_content(resp.choices[0].message.content)
                last_err = None
                break
            except Exception as e:
                last_err = str(e)[:300]
                time.sleep(2)

        decisions = parse_decisions(out_text) if not last_err else []

        for j, row in enumerate(buf, start=1):
            dec = "drop" if last_err else (decisions[j-1] if j-1 < len(decisions) else "drop")
            if dec == "keep":
                yield (
                    row["txt_id"],
                    row["txt_path"],
                    row["chunk_id"],
                    row["chunk_text"],
                    row["shard_id"],
                )

        buf = []

    # leftover
    if buf:
        items = [{"i": j+1, "text": buf[j]["chunk_text"]} for j in range(len(buf))]
        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": build_user_prompt(items)},
        ]

        last_err = None
        out_text = ""
        for _ in range(6):
            try:
                resp = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=messages,
                    max_tokens=MAX_OUT,
                    temperature=TEMPERATURE,
                )
                out_text = extract_text_from_content(resp.choices[0].message.content)
                last_err = None
                break
            except Exception as e:
                last_err = str(e)[:300]
                time.sleep(2)

        decisions = parse_decisions(out_text) if not last_err else []

        for j, row in enumerate(buf, start=1):
            dec = "drop" if last_err else (decisions[j-1] if j-1 < len(decisions) else "drop")
            if dec == "keep":
                yield (
                    row["txt_id"],
                    row["txt_path"],
                    row["chunk_id"],
                    row["chunk_text"],
                    row["shard_id"],
                )

# ---------- run (one shard) ----------
df0 = (
    spark.table(SRC_CHUNK_TABLE)
    .where(F.col("shard_id") == F.lit(SHARD_ID))
    .select("txt_id", "txt_path", "chunk_id", "chunk_text", "shard_id")
)

print("stageC_count_this_shard =", df0.count())

rdd = df0.repartition(N_PARTITIONS).rdd.mapPartitions(llm_filter_partition)
df_keep = spark.createDataFrame(rdd, "txt_id string, txt_path string, chunk_id string, chunk_text string, shard_id int")

(df_keep.write
 .mode("append")
 .format("delta")
 .saveAsTable(OUT_KEEP_TABLE)
)

print("WROTE =", OUT_KEEP_TABLE)
print("kept_count_this_shard =", df_keep.count())
display(df_keep.limit(10))

# Save text chunk file from Datalake to DBX volume

In [0]:
# Databricks notebook cell 1
# Goal: list first N txt files under SRC_DIR, and preview one file to confirm ABFSS access.

SRC_DIR = "abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/converted_raw_txts/"
N_TXT = 100  # <--- change this later

# List txt paths
all_txt = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.txt")
    .load(SRC_DIR)
    .select("path")
    .orderBy("path")
)

txt_paths = [r["path"] for r in all_txt.limit(N_TXT).collect()]
print("N_TXT =", N_TXT)
print("found =", len(txt_paths))
print("first 5 paths:")
for p in txt_paths[:5]:
    print("  ", p)

# Preview the first txt (first ~800 chars)
if txt_paths:
    first_path = txt_paths[0]
    preview = (
        spark.read.format("text")
        .load(first_path)
        .limit(2000)  # enough lines to preview
        .toPandas()["value"]
        .astype(str)
        .tolist()
    )
    preview_text = "\n".join(preview)
    print("\n---- preview file ----")
    print("path:", first_path)
    print(preview_text[:800])
else:
    print("No txt files found under SRC_DIR.")

In [0]:
# Databricks notebook cell 2
# Goal: apply file-level rules to the first N_TXT txt files and report pass/skip stats.
# Note: keep it self-contained (no custom functions import).

import re

# ---------- rules (same as your chunking_rules.py) ----------
DOMAIN_KEYWORDS_CORE = [
    "disaster", "hazard", "hazard mitigation",
    "flood", "flooding", "flash flood",
    "storm", "severe storm",
    "hurricane", "tropical storm", "tropical cyclone",
    "tornado", "wildfire",
    "winter storm", "freeze", "cold wave",
    "evacuation",
    "emergency operations plan", "eop",
    "hazard mitigation plan", "hmp",
    "after action report", "aar",
    "improvement plan",
    "state flood plan", "regional flood plan",
    "stormwater master plan", "mitigation plan",
    "power outage", "electric reliability", "grid",
    "critical infrastructure",
    "resilience", "resiliency",
    "emergency management",
]

DOMAIN_KEYWORDS_SUPPORT = [
    "watershed", "water quality", "cwa 319",
    "stormwater", "runoff",
    "nepa", "environmental impact statement", "categorical exclusion",
    "emergency declaration", "disaster declaration",
    "fema", "tdem", "twdb", "glo", "hcfcd",
    "emergency operations",
    "continuity of operations", "coop", "cog",
]

TEXAS_STRONG_TERMS = [
    "twdb", "tdem", "hcfcd", "txdot", "ercot",
    "texas commission on environmental quality", "tceq", "glo"
]

STRONG_DOMAIN_TERMS = [
    "flood", "flooding", "inundation",
    "stormwater", "watershed", "runoff", "drainage",
    "levee", "dam", "reservoir",
    "rainfall", "precipitation",
    "hurricane", "tornado", "wildfire", "winter storm", "drought",
    "evacuation", "emergency management", "hazard mitigation", "mitigation",
    "fema",
    "water quality", "groundwater", "aquifer",
    "wastewater", "sewer", "treatment plant",
    "erosion", "sediment",
    "superfund", "epa", "contamination",
    "critical infrastructure", "power outage", "grid reliability", "telecommunications"
]

EVENT_CORE_TERMS = [
    "flood", "flooding", "flash flood",
    "hurricane", "tropical storm", "tropical cyclone",
    "tornado",
    "wildfire",
    "winter storm", "freeze", "cold wave",
    "drought",
    "heat wave", "heatwave",
    "storm surge",
]

EVENT_CONTEXT_TERMS = [
    "fema",
    "evacuation", "shelter", "sheltering",
    "warning", "watch", "advisory",
    "disaster declaration", "emergency declaration",
    "after action report", "aar",
    "hazard mitigation plan", "hmp",
    "damage", "loss", "payout", "recovery",
    "inundation", "stormwater", "watershed", "runoff",
    "rainfall", "precipitation", "streamflow", "surge",
    "floodplain", "nfip", "flood insurance",
]

FILE_BLACKLIST_TERMS = [
    "touchdowns", "receiving yards", "rushing yards", "3pt", "box score",
    "staar", "bullying", "fluency checks", "instructional coach", "curriculum",
    "grades 2-4", "grades 9-12", "student achievement", "campus meets or exceeds",
    "biography:", "curriculum vitae", "cv:",
    "roundtable:", "chair:", "comment:", "location:", "omni",
    "see fish and fisheries", "fisheries, see",
]

def clean_text(text: str) -> str:
    # Remove image placeholders, compress underscores and whitespace.
    if not text:
        return ""
    t = re.sub(r"<image:[^>]+>", " ", text)
    t = re.sub(r"_+", "____", t)
    t = " ".join(t.split())
    return t

def is_domain_relevant_text(text: str, core_min_hits: int = 1, support_min_hits: int = 4) -> bool:
    if not text:
        return False
    t = text.lower()
    if sum(1 for kw in DOMAIN_KEYWORDS_CORE if kw in t) >= core_min_hits:
        return True
    return sum(1 for kw in DOMAIN_KEYWORDS_SUPPORT if kw in t) >= support_min_hits

def is_texas_related_strong(text: str) -> bool:
    if not text:
        return False
    t = text.lower()
    if re.search(r"\btexas\b", t):
        return True
    return any(k in t for k in TEXAS_STRONG_TERMS)

def has_strong_domain_signal(text: str, min_hits: int = 1) -> bool:
    if not text:
        return False
    t = text.lower()
    hits = sum(1 for kw in STRONG_DOMAIN_TERMS if kw in t)
    return hits >= min_hits

def disaster_gate_file(text: str) -> bool:
    if not text:
        return False
    t = text.lower()
    if any(b in t for b in FILE_BLACKLIST_TERMS):
        return False
    core_ok = any(k in t for k in EVENT_CORE_TERMS)
    ctx_ok = any(k in t for k in EVENT_CONTEXT_TERMS)
    return core_ok and ctx_ok

def file_decision(raw_text: str, texas_only: bool = True, disaster_only: bool = True):
    # Return (passed: bool, reason: str)
    text = clean_text(raw_text)
    if not text:
        return False, "EMPTY"

    if not is_domain_relevant_text(text):
        return False, "FAIL_domain_relevant"

    if texas_only and (not is_texas_related_strong(text)):
        return False, "FAIL_texas_strong"

    if not has_strong_domain_signal(text, min_hits=1):
        return False, "FAIL_strong_domain"

    if disaster_only and (not disaster_gate_file(text)):
        return False, "FAIL_disaster_gate_file"

    return True, "PASS"

# ---------- run on your txt_paths from Cell 1 ----------
TEXAS_ONLY = True
DISASTER_ONLY = True
MAX_LINES_READ = 4000  # avoid collecting huge files; enough for rules

passed_paths = []
skipped_paths = []
skip_reasons = {}   # reason -> count
debug_rows = []     # a few examples for inspection

for p in txt_paths:
    lines = spark.read.format("text").load(p).limit(MAX_LINES_READ).toPandas()["value"].astype(str).tolist()
    raw = "\n".join(lines)

    ok, reason = file_decision(raw, texas_only=TEXAS_ONLY, disaster_only=DISASTER_ONLY)

    if ok:
        passed_paths.append(p)
        if len(debug_rows) < 6:
            debug_rows.append(("PASS", reason, p, raw[:250].replace("\n", " ")))
    else:
        skipped_paths.append(p)
        skip_reasons[reason] = skip_reasons.get(reason, 0) + 1
        if len(debug_rows) < 12:
            debug_rows.append(("SKIP", reason, p, raw[:250].replace("\n", " ")))

print("Total:", len(txt_paths))
print("Passed:", len(passed_paths))
print("Skipped:", len(skipped_paths))
print("\nSkip reasons:")
for k in sorted(skip_reasons, key=lambda x: (-skip_reasons[x], x)):
    print(f"  {k}: {skip_reasons[k]}")

print("\nDebug samples (first few):")
for tag, reason, p, head in debug_rows:
    print("\n---", tag, reason, "---")
    print("path:", p)
    print("head:", head)

In [0]:
# Databricks notebook cell 3
# Goal:
# - Chunk ONLY passed_paths from Cell 2
# - Sentence split by (. ! ? ;)
# - Merge into chunks with <= 1000 chars
# - Output columns: txt_id, txt_path, chunk_id, chunk_text
# - Save as Delta table under tdis_dev_data_catalog.tdir.rag_files

import re
import uuid
from typing import List

# -------- params --------
MAX_LEN = 1000
OUT_TABLE = "tdis_dev_data_catalog.tdir.stage0_chunked_100txt"  # <--- rename if you want

# -------- chunking helpers --------
def split_into_sentences_simple(text: str) -> List[str]:
    # Split on . ! ? ;  (keep it simple; strip spaces)
    t = " ".join((text or "").split())
    if not t:
        return []
    parts = re.split(r"(?<=[\.\!\?\;])\s+", t)
    return [p.strip() for p in parts if p and p.strip()]

def sentences_to_chunks(sentences: List[str], max_len: int = 1000) -> List[str]:
    chunks = []
    cur = ""
    for s in sentences:
        if not cur:
            if len(s) <= max_len:
                cur = s
            else:
                # Hard split long sentence
                for i in range(0, len(s), max_len):
                    piece = s[i:i+max_len]
                    if len(piece) == max_len:
                        chunks.append(piece)
                    else:
                        cur = piece
            continue

        cand = cur + " " + s
        if len(cand) <= max_len:
            cur = cand
        else:
            chunks.append(cur)
            cur = ""
            if len(s) <= max_len:
                cur = s
            else:
                for i in range(0, len(s), max_len):
                    piece = s[i:i+max_len]
                    if len(piece) == max_len:
                        chunks.append(piece)
                    else:
                        cur = piece

    if cur:
        chunks.append(cur)
    return chunks

def read_txt_to_string(path: str, max_lines: int = 200000) -> str:
    # Read text file into a single string
    lines = spark.read.format("text").load(path).limit(max_lines).toPandas()["value"].astype(str).tolist()
    return "\n".join(lines)

# -------- build rows --------
rows = []
for txt_path in passed_paths:
    raw = read_txt_to_string(txt_path)
    # Reuse your clean_text from Cell 2 (already defined there)
    cleaned = clean_text(raw)

    txt_id = str(uuid.uuid4())
    sents = split_into_sentences_simple(cleaned)
    chunks = sentences_to_chunks(sents, max_len=MAX_LEN)

    for j, c in enumerate(chunks):
        rows.append((
            txt_id,
            txt_path,
            f"{txt_id}_{j}",   # chunk_id
            c                 # chunk_text
        ))

print("passed_paths =", len(passed_paths))
print("total_chunks =", len(rows))

# -------- write to Delta table --------
df_out = spark.createDataFrame(rows, "txt_id string, txt_path string, chunk_id string, chunk_text string")

(df_out.write
  .mode("overwrite")
  .format("delta")
  .option("mergeSchema", "true")
  .saveAsTable(OUT_TABLE)
)

display(spark.table(OUT_TABLE).limit(10))

In [0]:
# Databricks notebook cell 5
# Goal:
# - Read stage0 chunk table
# - Distributed LLM decide KEEP/DROP in Spark mapPartitions (no concurrent futures)
# - Save KEEP rows to a new Delta table

import re
import time
import yaml
from openai import OpenAI

# ---------- params ----------
SRC_TABLE = "tdis_dev_data_catalog.tdir.stage0_chunked_100txt"
OUT_TABLE = "tdis_dev_data_catalog.tdir.stage1_llm_kept_100txt"

MODEL_NAME = "databricks-gpt-oss-20b"
BASE_URL = "https://adb-3300405005568038.18.azuredatabricks.net/serving-endpoints"
CONFIG_PATH = "../config.yaml"
TOKEN_KEY = "DATABRICKS_TOKEN"

BATCH = 64          # per LLM call
MAX_OUT = 1000       # max_tokens
TEMPERATURE = 0.0

# Controls total parallelism hitting endpoint (tune carefully)
N_PARTITIONS = 64   # start moderate; raise if endpoint allows

# ---------- disable mlflow autolog ----------
def disable_mlflow_autolog():
    # English comments only
    try:
        import mlflow
        try:
            import mlflow.openai
            mlflow.openai.autolog(disable=True)
        except Exception:
            pass
        try:
            mlflow.autolog(disable=True)
        except Exception:
            pass
    except Exception:
        pass

disable_mlflow_autolog()

# ---------- prompt + parsing (your style) ----------
def build_batch_decision_system_prompt() -> str:
    return (
        "For each chunk, output exactly one line: <i>\\tKEEP or <i>\\tDROP (i is the chunk number).\n"
        "DROP if the chunk is mainly ads/marketing/CTA, contact info, navigation/boilerplate, or mostly links/listing with little content.\n"
        "Otherwise KEEP.\n"
        "No extra text. Do not output reasoning."
    )

def build_batch_decision_user_prompt(items):
    parts = []
    for it in items:
        i = int(it["i"])
        txt = str(it["text"])
        parts.append(f"[{i}]\n{txt}")
    return "\n\n".join(parts)

def parse_decisions(text_out: str):
    pairs = []
    for ln in (text_out or "").splitlines():
        m = re.match(r"^\s*(\d+)\s*(?:\\t|\t|:|\s)\s*(KEEP|DROP)\s*$", ln, flags=re.I)
        if m:
            pairs.append((int(m.group(1)), m.group(2).lower()))
    pairs.sort(key=lambda x: x[0])
    return [d for _, d in pairs]

def extract_text_from_content(content):
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        for part in content:
            if isinstance(part, dict) and part.get("type") in ("text", "output_text"):
                if isinstance(part.get("text"), str) and part["text"].strip():
                    return part["text"]
        # fallback: reasoning summary
        for part in content:
            if isinstance(part, dict) and part.get("summary"):
                summ = part.get("summary")
                if isinstance(summ, list):
                    for s in summ:
                        if isinstance(s, dict) and s.get("type") in ("summary_text", "text"):
                            if isinstance(s.get("text"), str) and s["text"].strip():
                                return s["text"]
    return str(content)

def load_token(config_path: str, token_key: str) -> str:
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    return cfg[token_key]

# ---------- partition worker ----------
def llm_filter_partition(rows_iter):
    # English comments only
    disable_mlflow_autolog()

    token = load_token(CONFIG_PATH, TOKEN_KEY)
    client = OpenAI(api_key=token, base_url=BASE_URL)

    sys_prompt = build_batch_decision_system_prompt()

    buf = []
    for r in rows_iter:
        buf.append(r)
        if len(buf) < BATCH:
            continue

        items = [{"i": j+1, "text": buf[j]["chunk_text"]} for j in range(len(buf))]
        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": build_batch_decision_user_prompt(items)},
        ]

        # retry
        last_err = None
        out_text = ""
        for _ in range(6):
            try:
                resp = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=messages,
                    max_tokens=MAX_OUT,
                    temperature=TEMPERATURE,
                )
                out_text = extract_text_from_content(resp.choices[0].message.content)
                last_err = None
                break
            except Exception as e:
                last_err = str(e)[:300]
                time.sleep(2)

        decisions = parse_decisions(out_text) if not last_err else []

        for j, row in enumerate(buf, start=1):
            dec = "drop" if last_err else (decisions[j-1] if j-1 < len(decisions) else "drop")
            if dec == "keep":
                yield (
                    row["txt_id"],
                    row["txt_path"],
                    row["chunk_id"],
                    row["chunk_text"],
                )

        buf = []

    # leftover
    if buf:
        items = [{"i": j+1, "text": buf[j]["chunk_text"]} for j in range(len(buf))]
        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": build_batch_decision_user_prompt(items)},
        ]

        last_err = None
        out_text = ""
        for _ in range(6):
            try:
                resp = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=messages,
                    max_tokens=MAX_OUT,
                    temperature=TEMPERATURE,
                )
                out_text = extract_text_from_content(resp.choices[0].message.content)
                last_err = None
                break
            except Exception as e:
                last_err = str(e)[:300]
                time.sleep(2)

        decisions = parse_decisions(out_text) if not last_err else []

        for j, row in enumerate(buf, start=1):
            dec = "drop" if last_err else (decisions[j-1] if j-1 < len(decisions) else "drop")
            if dec == "keep":
                yield (
                    row["txt_id"],
                    row["txt_path"],
                    row["chunk_id"],
                    row["chunk_text"],
                )

# ---------- run distributed ----------
df0 = spark.table(SRC_TABLE).select("txt_id", "txt_path", "chunk_id", "chunk_text")
print("stage0_count =", df0.count())

rdd = df0.repartition(N_PARTITIONS).rdd.mapPartitions(llm_filter_partition)
df_keep = spark.createDataFrame(rdd, "txt_id string, txt_path string, chunk_id string, chunk_text string")

(df_keep.write
 .mode("overwrite")
 .format("delta")
 .option("mergeSchema", "true")
 .saveAsTable(OUT_TABLE)
)

print("kept_count =", spark.table(OUT_TABLE).count())
display(spark.table(OUT_TABLE).limit(10))

In [0]:
# Databricks notebook cell
# Stage2: Embed kept chunks with DMRetriever-33M (CPU) + Create Databricks Vector Search index
# Input table : tdis_dev_data_catalog.tdir.stage1_llm_kept_100txt
# Output table: tdis_dev_data_catalog.tdir.cleaned_rag_chunks (with embedding)
# VS index    : tdis_dev_data_catalog.tdir.cleaned_rag_chunks_vs

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

# ---- Config ----
SRC_KEEP_TABLE = "tdis_dev_data_catalog.tdir.stage1_llm_kept_100txt"
EMB_TABLE = "tdis_dev_data_catalog.tdir.cleaned_rag_chunks"          # delta table to hold vectors
VS_INDEX = "tdis_dev_data_catalog.tdir.cleaned_rag_chunks_vs"        # vector search index name

MODEL_ID = "DMIR01/DMRetriever-33M"
DEVICE = "cpu"
MAX_LENGTH = 512
BATCH = 128
N_PARTITIONS = 64   # adjust if needed

# ---- Mean pooling + L2 normalize (same idea as embed.py) ----
@torch.no_grad()
def mean_pooling(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)   # [B,T,1]
    summed = (last_hidden_state * mask).sum(dim=1)                   # [B,H]
    counts = mask.sum(dim=1).clamp(min=1e-9)                         # [B,1]
    return summed / counts

def embed_partition(rows_iter):
    # Load model on executor
    tok = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
    mdl = AutoModel.from_pretrained(MODEL_ID)
    mdl.to(DEVICE)
    mdl.eval()

    buf = []
    meta = []
    for r in rows_iter:
        buf.append(r["chunk_text"])
        meta.append((r["txt_id"], r["txt_path"], r["chunk_id"], r["chunk_text"]))
        if len(buf) < BATCH:
            continue

        enc = tok(buf, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
        out = mdl(**{k: v.to(DEVICE) for k, v in enc.items()})
        emb = mean_pooling(out.last_hidden_state, enc["attention_mask"].to(DEVICE))
        emb = torch.nn.functional.normalize(emb, p=2, dim=1).cpu().numpy().astype(np.float32)

        for (txt_id, txt_path, chunk_id, chunk_text), vec in zip(meta, emb):
            yield (txt_id, txt_path, chunk_id, chunk_text, vec.tolist())

        buf, meta = [], []

    # leftover
    if buf:
        enc = tok(buf, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
        out = mdl(**{k: v.to(DEVICE) for k, v in enc.items()})
        emb = mean_pooling(out.last_hidden_state, enc["attention_mask"].to(DEVICE))
        emb = torch.nn.functional.normalize(emb, p=2, dim=1).cpu().numpy().astype(np.float32)

        for (txt_id, txt_path, chunk_id, chunk_text), vec in zip(meta, emb):
            yield (txt_id, txt_path, chunk_id, chunk_text, vec.tolist())

# ---- Load kept chunks ----
df = spark.table(SRC_KEEP_TABLE).select("txt_id", "txt_path", "chunk_id", "chunk_text")
print("kept_count =", df.count())

# ---- Embed on cluster ----
rdd = df.repartition(N_PARTITIONS).rdd.mapPartitions(embed_partition)
schema = """
txt_id string,
txt_path string,
chunk_id string,
chunk_text string,
embedding array<float>
"""
df_emb = spark.createDataFrame(rdd, schema)

# ---- Write embeddings to Delta table ----
(df_emb.write
 .mode("overwrite")
 .format("delta")
 .option("mergeSchema", "true")
 .saveAsTable(EMB_TABLE)
)

print("emb_table_count =", spark.table(EMB_TABLE).count())

In [0]:
# Databricks notebook cell
# Read chunks.jsonl from ADLS and write Delta files into UC Volume as raw_chunks (new address)

base = "abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/"
dst  = "/Volumes/tdis_dev_data_catalog/tdir/rag_files/raw_chunks"  # <-- raw_chunks in the volume

df_chunks = (
    spark.read.json(base + "optimized_chunks/*/chunks.jsonl")
    .select("chunk_id", "source_file", "chunk_index_in_file", "text")
)

df_chunks.write.mode("overwrite").format("delta").save(dst)

display(spark.read.format("delta").load(dst).limit(5))

# Filter and clean text chunks using LLMs

In [0]:
# Databricks notebook cell
# Full run (no hard-drop):
# - Read ALL raw_chunks
# - Batch LLM decide KEEP/DROP
# - Save ONLY KEEP rows with cleaned text to /Volumes/.../cleaned_chunks (Delta overwrite)

import pandas as pd
import re, time
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import importlib

import functions.clean as clean
import functions.LLM_call as L
importlib.reload(clean)
importlib.reload(L)

SRC = "/Volumes/tdis_dev_data_catalog/tdir/rag_files/raw_chunks"
DST = "/Volumes/tdis_dev_data_catalog/tdir/rag_files/cleaned_chunks"
ENDPOINT_NAME = "RAG_chunk_cleaner"

BATCH = 48
WORKERS = 8
MAX_OUT = 500
FLUSH_EVERY = 20000

def parse_decisions(text_out: str) -> list[str]:
    # Return decisions in order, e.g., ["keep","drop",...]
    pairs = []
    for ln in (text_out or "").splitlines():
        m = re.match(r"^\s*(\d+)\s*(?:\\t|\t|:|\s)\s*(KEEP|DROP)\s*$", ln, flags=re.I)
        if m:
            pairs.append((int(m.group(1)), m.group(2).lower()))
    pairs.sort(key=lambda x: x[0])
    return [d for _, d in pairs]

def _call_llm_for_items(items):
    msgs = clean.build_batch_decision_messages(items)
    last_err = ""
    for _ in range(6):
        try:
            text_out = L.llm_chat(
                messages=msgs,
                model_name=ENDPOINT_NAME,
                max_tokens=MAX_OUT,
                temperature=0.2,
                config_path="../config.yaml",
            )
            return parse_decisions(text_out), ""
        except Exception as e:
            last_err = str(e)[:300]
            time.sleep(2)
    return [], last_err

# Load all rows (stream to driver)
df = (spark.read.format("delta").load(SRC)
      .select("chunk_id","source_file","chunk_index_in_file","text"))
total = df.count()
print("total_raw_chunks =", total)

# Init output (overwrite)
spark.createDataFrame([], "chunk_id string, source_file string, chunk_index_in_file long, text string") \
     .write.mode("overwrite").format("delta").save(DST)

it = df.toLocalIterator()
out_buf = []

def _flush_out(buf):
    if not buf:
        return
    spark.createDataFrame(pd.DataFrame(buf, columns=["chunk_id","source_file","chunk_index_in_file","text"])) \
         .write.mode("append").format("delta").save(DST)
    buf.clear()

with ThreadPoolExecutor(max_workers=WORKERS) as ex, tqdm(total=total) as pbar:
    pending = []   # (future, rows)
    batch_rows = []

    def submit_batch(rows):
        items = [{"i": j+1, "text": str(r["text"])} for j, r in enumerate(rows)]
        fut = ex.submit(_call_llm_for_items, items)  # returns (decisions_list, err)
        return (fut, rows)

    # Fill initial queue
    while len(pending) < WORKERS:
        try:
            r = next(it).asDict()
        except StopIteration:
            break
        batch_rows.append(r)
        if len(batch_rows) == BATCH:
            pending.append(submit_batch(batch_rows))
            batch_rows = []

    # Main loop
    while pending:
        done = [p for p in pending if p[0].done()]
        if not done:
            time.sleep(0.01)
            continue

        for fut, rows in done:
            pending.remove((fut, rows))
            decisions, err = fut.result()

            for j, r in enumerate(rows, start=1):
                dec = "ERROR" if err else (decisions[j-1].upper() if j-1 < len(decisions) else "DROP")
                if dec == "KEEP":
                    out_buf.append((
                        str(r["chunk_id"]),
                        r["source_file"],
                        int(r["chunk_index_in_file"]),
                        clean.clean_text_keep(r["text"]),
                    ))
                pbar.update(1)

            if len(out_buf) >= FLUSH_EVERY:
                _flush_out(out_buf)

            # Refill queue
            while len(pending) < WORKERS:
                try:
                    r = next(it).asDict()
                except StopIteration:
                    break
                batch_rows.append(r)
                if len(batch_rows) == BATCH:
                    pending.append(submit_batch(batch_rows))
                    batch_rows = []

    # leftover partial batch
    if batch_rows:
        decisions, err = _call_llm_for_items([{"i": j+1, "text": str(r["text"])} for j, r in enumerate(batch_rows)])
        for j, r in enumerate(batch_rows, start=1):
            dec = "ERROR" if err else (decisions[j-1].upper() if j-1 < len(decisions) else "DROP")
            if dec == "KEEP":
                out_buf.append((
                    str(r["chunk_id"]),
                    r["source_file"],
                    int(r["chunk_index_in_file"]),
                    clean.clean_text_keep(r["text"]),
                ))
            pbar.update(1)

    _flush_out(out_buf)

display(spark.read.format("delta").load(DST).limit(5))

In [0]:
spark.read.format("delta").load("/Volumes/tdis_dev_data_catalog/tdir/rag_files/cleaned_chunks").count()

# Embed